# pr_runner — IEC 61724-2 PR Test

Called by `calculate_pr` (or directly by the Data Factory pipeline) after
`scada_ingest` and `csv_ingest` have populated the Silver tables.

**Parameters:**
- `plant_code` — e.g. `PL1`
- `period_start` — ISO timestamp, e.g. `2026-02-16 00:00:00`
- `period_end`   — ISO timestamp
- `run_id`       — pipeline run identifier

In [7]:
plant_code  = ""
period_start = ""
period_end   = ""
run_id       = ""


In [8]:
# --- Publicación del HTML report en SharePoint vía Microsoft Graph ---
publish_html_to_sharepoint = True

# --- App registrada en Azure AD: "PVSyst1_EPI" ---
# client_id y tenant_id son identificadores públicos (no son secretos).
client_id  = "58f369e5-0dfd-4435-802b-a90a6786bf22"
tenant_id  = "943fa85b-068d-404a-bacb-5f0baf8d31a8"

# El client_secret SÍ es sensible: vive en Key Vault.
key_vault_url             = "https://apicredentialsenerland.vault.azure.net/"
secret_client_secret_name = "PVSyst1EPISecret"

# Destino SharePoint
sp_site_path    = "enerlandgroup.sharepoint.com:/sites/Pvsyst1"
sp_drive_name   = "Documentos"     # nombre real de la biblioteca en el site
sp_reports_path = "PERFORMANCE"

## 1 · Configuration

In [9]:
import notebookutils
from pathlib import Path
import datetime as dt
import json
import pandas as pd
import shutil

LAKEHOUSE_FILES      = '/lakehouse/default/Files'
LAKEHOUSE_CONFIG_DIR = f'{LAKEHOUSE_FILES}/Raw PVSyst1/CONFIG'
LAKEHOUSE_PERF_DIR   = f'{LAKEHOUSE_FILES}/Raw PVSyst1/PERFORMANCE'
CONFIG_XLSX_TMP      = '/tmp/plants_config_pr.xlsx'
REPORT_TEMPLATE_TMP  = '/tmp/report_template.py'
LOGO_B64_TMP         = '/tmp/_logo_b64.py'


def _copy_if_missing(src_path, local_path):
    """Copia desde el lakehouse local (/lakehouse/default/Files/...) a /tmp si no existe."""
    if Path(local_path).exists():
        return True
    src = Path(src_path)
    if src.exists():
        shutil.copy2(str(src), local_path)
        return True
    print(f"WARN: not found: {src_path}")
    return False


def _read_cfg_table(sheet, table=None):
    """Lee una TABLA ESTRUCTURADA del Excel de config: 1) por NOMBRE de tabla
    (case-insensitive); 2) si no, la (primera) tabla de la hoja; 3) hoja completa."""
    _copy_if_missing(f'{LAKEHOUSE_CONFIG_DIR}/plants_config.xlsx', CONFIG_XLSX_TMP)
    import openpyxl
    wb = openpyxl.load_workbook(CONFIG_XLSX_TMP, data_only=True)
    def _df(ws, ref):
        _d = [[c.value for c in _r] for _r in ws[ref]]
        return pd.DataFrame(_d[1:], columns=_d[0]) if _d else pd.DataFrame()
    if table:
        for ws in wb.worksheets:
            _tm = {k.lower(): k for k in ws.tables}
            if table.lower() in _tm:
                return _df(ws, ws.tables[_tm[table.lower()]].ref)
    if sheet in wb.sheetnames:
        ws = wb[sheet]
        if ws.tables:
            return _df(ws, ws.tables[next(iter(ws.tables))].ref)
    return pd.read_excel(CONFIG_XLSX_TMP, sheet_name=sheet)
def _active_criteria(plant_code):
    """04_Criteria_Selection -> set de criterios ACTIVOS de la planta.
    c1-c7 activos si valor >= 0 (-1 = off); c8-c9 activos si TRUE."""
    act = set()
    try:
        sc = _read_cfg_table('04_Criteria_Selection', 'T04_criteria_selection')
        sc.columns = [str(c).strip().lower() for c in sc.columns]
        sub = sc[sc['plant_code'].astype(str).str.strip() == plant_code]
        if not sub.empty:
            row = sub.iloc[0]
            for n in range(1, 8):
                ck = f'c{n}'
                if ck in sc.columns:
                    try:
                        if float(row[ck]) >= 0:
                            act.add(ck)
                    except (ValueError, TypeError):
                        pass
            for n in (8, 9):
                ck = f'c{n}'
                if ck in sc.columns and str(row[ck]).strip().upper() in ('TRUE','VERDADERO','1','SI','SÍ','YES','Y'):
                    act.add(ck)
    except Exception:
        pass
    return act


def _epi_factors(plant_code):
    """03_EPI_Factors: base row (blank period) + optional period rows. Returns
    (base, periods): base={epi,deg,avail}; periods=[{start,end,epi,deg,avail}]
    (Timestamps). Rows with blank period_start/period_end are the base factors."""
    def _f(v, d):
        try:
            return float(v) if v is not None and not pd.isna(v) else d
        except (ValueError, TypeError):
            return d
    base = {'epi': 0.98, 'deg': 1.0, 'avail': 1.0}
    periods = []
    try:
        fdf = _read_cfg_table('03_EPI_Factors', 'T03_criteria_config')
        fdf.columns = [str(c).strip().lower() for c in fdf.columns]
        sub = fdf[fdf['plant_code'].astype(str).str.strip() == plant_code]
        for _, r in sub.iterrows():
            fac = {'epi': _f(r.get('guaranteed_epi'), base['epi']),
                   'deg': _f(r.get('degradation_factor'), 1.0),
                   'avail': _f(r.get('availability_factor'), 1.0)}
            ps = pd.to_datetime(r.get('period_start'), errors='coerce')
            pe = pd.to_datetime(r.get('period_end'), errors='coerce')
            if pd.isna(ps) and pd.isna(pe):
                base = fac
            else:
                periods.append({'start': ps, 'end': pe, **fac})
    except Exception:
        pass
    return base, periods


def _guar_factor_series(date_s, contract):
    """Per-row guaranteed factor epi*deg*avail. Uses the EPI-factor period whose
    [start, end] range (inclusive, day-level) contains each date; base if none."""
    d = pd.to_datetime(date_s).dt.normalize()
    epi   = pd.Series(float(contract['guaranteed_epi']),           index=date_s.index)
    deg   = pd.Series(float(contract.get('degradation_factor', 1.0)),  index=date_s.index)
    avail = pd.Series(float(contract.get('availability_factor', 1.0)), index=date_s.index)
    for p in contract.get('epi_periods', []):
        m = pd.Series(True, index=date_s.index)
        if pd.notna(p.get('start')): m &= d >= pd.Timestamp(p['start']).normalize()
        if pd.notna(p.get('end')):   m &= d <= pd.Timestamp(p['end']).normalize()
        epi[m] = p['epi']; deg[m] = p['deg']; avail[m] = p['avail']
    return epi * deg * avail


def load_plant_config(plant_code: str) -> dict:
    _copy_if_missing(f'{LAKEHOUSE_CONFIG_DIR}/plants_config.xlsx', CONFIG_XLSX_TMP)
    plants   = _read_cfg_table('01_Plants', 'T01_plants')
    mapping  = _read_cfg_table('02_Sensors_Mapping', 'T02_sensor_mapping')
    albedo = pd.DataFrame()  # albedo se calcula en scada_proc (ALB_avg_filt); no hay hoja de pares aqui

    prow = plants[plants.get('plant_code', pd.Series(index=plants.index)).fillna('') == plant_code]
    if prow.empty:
        raise ValueError(f'{plant_code} not found in 01_Plants (columna plant_code)')
    prow = prow.iloc[0]

    epi_base, epi_periods = _epi_factors(plant_code)

    agg_rules = {}
    if 'plant_code' in mapping.columns:
        _mapping_rows = mapping[mapping['plant_code'] == plant_code]
        if _mapping_rows.empty:
            _mapping_rows = mapping[mapping['plant_code'] == 'COMMON']
    else:
        # Si no hay columna plant_code en el mapping, asumimos que todas las filas aplican como COMMON
        _mapping_rows = mapping.copy()

    for _, r in _mapping_rows.iterrows():
        if pd.notna(r.get('columna_destino')) and pd.notna(r.get('agregacion')):
            dest = str(r['columna_destino']).strip()
            if dest != 'Date':
                agg_rules[dest] = str(r['agregacion']).strip()

    _energy_src = next((str(r['columna_original']).strip()
                        for _, r in _mapping_rows.iterrows()
                        if str(r.get('columna_destino', '')).strip() == 'E_GRID'
                        and pd.notna(r.get('columna_original'))), 'E_GRID')

    albedo_pairs = []
    if not albedo.empty:
        if 'plant_code' in albedo.columns:
            _alb_rows = albedo[albedo['plant_code'] == plant_code]
        else:
            _alb_rows = albedo.copy()
        for _, r in _alb_rows.iterrows():
            if pd.notna(r.get('ref_column')) and pd.notna(r.get('poa_column')):
                albedo_pairs.append({
                    'label': str(r.get('alb_label', '')).strip(),
                    'ref':   str(r['ref_column']).strip(),
                    'poa':   str(r['poa_column']).strip(),
                })

    return {
        'plant': {
            'code':           plant_code,
            'name':           str(prow.get('plant_name', plant_code)),
            'ac_capacity_kw': float(prow.get('ac_capacity_kw', 0)),
            'dc_capacity_kw': float(prow.get('dc_capacity_kw', 0)),
            'installed_mwp':  float(prow.get('dc_capacity_kw', 0)) / 1000,
        },
        'contract': {
            # Base EPI factors (03_EPI_Factors, blank-period row); per-period overrides in epi_periods.
            'guaranteed_epi':           epi_base['epi'],
            'degradation_factor':       epi_base['deg'],
            'availability_factor':      epi_base['avail'],
            'epi_periods':              epi_periods,
            'energy_ref_sensor':        _energy_src,   # energy always from the E_GRID mapped column
            'aggregated_hours_min':     0.0,
            'aggregate_by_hour':        str(prow.get('aggregate_by_hour', '')).strip().upper() in ('TRUE', 'VERDADERO', '1', 'SI', 'YES'),
        },
        'active_criteria':   _active_criteria(plant_code),
        'aggregation_rules': agg_rules,
        'albedo_pairs':      albedo_pairs,
    }


cfg = load_plant_config(plant_code)
print(f"Plant : {cfg['plant']['name']} ({plant_code})")
print(f"Period: {period_start} -> {period_end}")
# ---------------------------------------------------------
# Time-shift T08 — se aplica al SCADA, ANTES de agrupar
# ---------------------------------------------------------
# `time_shift` de T08_time_shifts son los minutos que hay que SUMAR al lado
# PVsyst, es decir a la energia esperada y a la garantizada. Es el convenio de
# siempre: hasta que el desfase se movio al SCADA, csv_ingest hacia
# `proc_pvsyst.Date = Date + time_shift` (asi se valido FRA, 121% -> 104,3%).
#
# Sumar X a PVsyst es lo mismo que RESTAR X al SCADA, y restarlo al SCADA es lo
# que se hace aqui, porque el SCADA es la serie fina y el desfase (75 min de
# PL1) no cabe en la rejilla horaria de PVsyst. Tras casar las dos series, en
# compute_pr_test se devuelve la fecha a HORA REAL sumando el mismo desfase:
# el resultado neto es el desplazamiento aplicado a esperada y garantizada.
#
# Se redondea a la resolucion NATIVA del raw SCADA (15 min en estas plantas):
# con un desfase que no cae en esa rejilla, el resample posterior mandaria unas
# filas al bucket de arriba y otras al de abajo. Redondeando a la rejilla el
# agrupado es determinista y el desfase sobrevive entero al bucket horario (los
# 75 min de PL1 siguen siendo 75, no 60).
#
# Al ser multiplo de la rejilla, el SCADA desplazado SIGUE siendo una rejilla de
# 15 min y cada bucket horario conserva sus 4 intervalos: solo quedan buckets
# parciales en los bordes del periodo y en la costura del cambio de hora.
# Lo que si deja un desfase que no es multiplo de 60 (p.ej. los 75 de PL1) es que
# uno de los 4 cuartos de cada hora de PVsyst cae en el bucket de al lado: se
# alinea mejor la hora solar a cambio de mezclar un cuarto con la hora vecina.
#
# Semantica de periodos (la misma que documenta T08):
#   - Lo que no cubre ningun periodo queda a 0.
#   - end_date es INCLUSIVO, y las costuras caen a las DST_SEAM_HOUR del dia
#     indicado: el reloj en la UE cambia a las 02:00/03:00 locales, no a
#     medianoche, asi que cortar a dia completo desalinearia la madrugada.
#   - Puede haber VARIOS periodos por planta; si se solapan gana el de MAS
#     ABAJO en la tabla (las reglas se aplican en orden).
#   - Un 0 explicito es una regla valida: anula un periodo anterior solapado.
#   - Fecha vacia = sin limite por ese lado.
DST_SEAM_HOUR = 2


def t08_rules(plant_code):
    """T08_time_shifts -> [(start, end, minutos)] de la planta, EN EL ORDEN DE LA TABLA."""
    rules = []
    try:
        df = _read_cfg_table('08_Time_Shifts', 'T08_time_shifts')
        df.columns = [str(c).strip().lower() for c in df.columns]
        sub = df[df['plant_code'].astype(str).str.strip().str.upper()
                 == str(plant_code).strip().upper()]
        for _, r in sub.iterrows():
            try:
                raw = float(r.get('time_shift'))
            except (TypeError, ValueError):
                continue          # celda vacia o no numerica: no es una regla
            if pd.isna(raw):
                continue
            start = pd.to_datetime(r.get('start_date'), errors='coerce')
            end   = pd.to_datetime(r.get('end_date'),   errors='coerce')
            rules.append((None if pd.isna(start) else start,
                          None if pd.isna(end)   else end,
                          raw))
    except Exception as e:
        print(f'  [time_shift] no se pudieron leer reglas T08 para {plant_code}: {e}')
    return rules


def scada_time_shift(dates, plant_code, scada_res):
    """Minutos de desfase de T08, con el signo de T08 (los que se le suman a PVsyst).

    El llamante los RESTA a las fechas del SCADA, que es la operacion inversa y
    equivalente, y los vuelve a sumar despues del merge para dejar el resultado
    en hora real.

    `scada_res` es la resolucion nativa del raw SCADA y define la rejilla a la
    que se redondea el desfase. Devuelve una Serie int64 alineada con `dates`
    (0 en las filas que no cubre ninguna regla).
    """
    step = max(1, int(round(pd.Timedelta(scada_res).total_seconds() / 60)))
    out  = pd.Series(0, index=dates.index, dtype='int64')
    seam = pd.Timedelta(hours=DST_SEAM_HOUR)
    for start, end, raw in t08_rules(plant_code):
        mins = int(round(raw / step)) * step
        mask = pd.Series(True, index=dates.index)
        if start is not None:
            mask &= dates > pd.Timestamp(start).normalize() + seam
        if end is not None:
            mask &= dates <= pd.Timestamp(end).normalize() + seam
        out.loc[mask] = mins
    return out


## 2 · Read Processed tables (filtered by plant & period)

In [10]:
# Resolve period if not provided
if not period_start or not period_end:
    row = spark.sql(f"""
        SELECT MIN(Date) AS mn, MAX(Date) AS mx
        FROM scada_proc
        WHERE plant_code = '{plant_code}'
    """).first()
    if row['mn'] is None:
        raise ValueError(f'No SCADA data found for {plant_code} en scada_proc')
    period_start = str(row['mn'])
    period_end   = str(row['mx'])

print(f"Period: {period_start} -> {period_end}")

scada_sdf = spark.sql(f"""
    SELECT *
    FROM scada_proc
    WHERE plant_code = '{plant_code}'
      AND Date BETWEEN TIMESTAMP'{period_start}' AND TIMESTAMP'{period_end}'
    ORDER BY Date
""")
pvsyst_sdf = spark.sql(f"""
    SELECT *
    FROM proc_pvsyst
    WHERE plant_code = '{plant_code}'
      AND Date BETWEEN TIMESTAMP'{period_start}' AND TIMESTAMP'{period_end}'
    ORDER BY Date
""")

scada  = scada_sdf.toPandas().drop(columns=['plant_code','source_file','ingested_at','processed_at'], errors='ignore')
pvsyst = pvsyst_sdf.toPandas().drop(columns=['plant_code','source_file','ingested_at'], errors='ignore')


if len(scada) == 0:
    raise ValueError(f'No SCADA data for {plant_code} in {period_start} -> {period_end}')
if len(pvsyst) == 0:
    raise ValueError(f'No PVSyst data for {plant_code} in {period_start} -> {period_end}')

scada_res  = scada['Date'].diff().dropna().mode().iloc[0]
pvsyst_res = pvsyst['Date'].diff().dropna().mode().iloc[0]
print(f'SCADA (proc): {len(scada):,} rows @ {scada_res}')
print(f'PVSyst:       {len(pvsyst):,} rows @ {pvsyst_res}')
# Time-shift T08: aqui solo se ANOTA (columna _shift_min, en la rejilla nativa
# del SCADA). El desplazamiento se aplica mas abajo, despues de evaluar los
# criterios, para que el agrupado por dia de c8/c9 y los dias excluidos sigan
# calculandose en hora real. PVsyst se queda tal como lo entrega el CSV.
_shift_min = scada_time_shift(scada['Date'], plant_code, scada_res)
scada['_shift_min'] = _shift_min.values
_tramos = sorted({int(x) for x in _shift_min.unique()})
if any(_t != 0 for _t in _tramos):
    _n = int((_shift_min != 0).sum())
    print(f'Time-shift T08: {_tramos} min a sumar a PVsyst (rejilla {scada_res}) '
          f'en {_n:,}/{len(scada):,} filas')
else:
    print('Time-shift T08: sin desfase declarado para esta planta/periodo')

## 3 · PR engine — IEC 61724-2 (9 criteria)

In [11]:
from dataclasses import dataclass, field

@dataclass
class CriteriaAudit:
    total_intervals: int = 0
    daytime_intervals: int = 0   # intervals with POA > 0 (used for % display)
    c1_failed: int = 0           # POA_avg_filt no disponible
    c2_failed: int = 0           # GHI_avg_filt no disponible
    c3_failed: int = 0           # DHI_avg_filt no disponible
    c4_failed: int = 0           # REF_avg_filt no disponible
    c5_failed: int = 0           # horas/dia validas de POA < umbral
    c6_failed: int = 0           # horas/dia validas de GHI < umbral
    c7_failed: int = 0           # setpoint por debajo del umbral (operador limita)
    c8_failed: int = 0           # Wind Alarm activa (WA == 1)
    c9_failed: int = 0           # PF fuera de rango (<=0.98 o >=1.0)
    valid_intervals: int = 0
    valid_days: int = 0
    excluded_days: list = field(default_factory=list)
    aggregated_valid_hours: float = 0.0
    def as_dict(self): return self.__dict__


def _col(df, name):
    return df[name].fillna(0) if name in df.columns else pd.Series(0.0, index=df.index)


def evaluate_criteria(scada, cfg):
    """Criterios c1-c9 precomputados en scada_proc. Lee all_valid + cn columnas.
    Construye el audit y pasa los flags al resto del pipeline."""
    c = cfg['contract']
    df = scada.copy()
    df['_day'] = df['Date'].dt.date

    if 'POA_avg_filt' in df.columns:
        df['POA_avg'] = df['POA_avg_filt']
    elif 'POA_avg' not in df.columns:
        df['POA_avg'] = float('nan')

    # Asegurar columnas de criterios (por si la tabla aun no tiene la columna)
    for k in ('c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'all_valid'):
        if k not in df.columns:
            df[k] = 1
        df[k] = df[k].fillna(0).astype(int)

    dt_h = (df['Date'].iloc[1] - df['Date'].iloc[0]).total_seconds() / 3600 if len(df) >= 2 else 0.25

    daytime_mask = df['POA_avg'].fillna(0) > 0
    if 'hours_aggregated' in df.columns:
        valid_hours_total = float(df['hours_aggregated'].sum())
    else:
        valid_hours_total = float(df['all_valid'].sum() * dt_h)

    c5_pass = valid_hours_total >= c.get('aggregated_hours_min', 0)

    audit = CriteriaAudit(
        total_intervals        = len(df),
        daytime_intervals      = int(daytime_mask.sum()),
        c1_failed              = int((df['c1'] == 0).sum()),
        c2_failed              = int((df['c2'] == 0).sum()),
        c3_failed              = int((df['c3'] == 0).sum()),
        c4_failed              = int((df['c4'] == 0).sum()),
        c5_failed              = int((df['c5'] == 0).sum()),
        c6_failed              = int((df['c6'] == 0).sum()),
        c7_failed              = int((df['c7'] == 0).sum()),
        c8_failed              = int((df['c8'] == 0).sum()),
        c9_failed              = int((df['c9'] == 0).sum()),
        valid_intervals        = int(df['all_valid'].sum()),
        aggregated_valid_hours = valid_hours_total,
    )
    valid_days = df.loc[df['all_valid'] == 1, '_day'].unique()
    audit.valid_days    = len(valid_days)
    audit.excluded_days = sorted(str(d) for d in df['_day'].unique() if d not in valid_days)

    for k in ('c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'all_valid'):
        if k in df.columns:
            df[k] = df[k].astype(bool)

    return df.drop(columns=['_day']), audit


def aggregate_to_target(scada_with_criteria, target_resolution, rules):
    native = scada_with_criteria['Date'].diff().dropna().mode().iloc[0]
    if target_resolution <= native:
        return scada_with_criteria
    freq = pd.tseries.frequencies.to_offset(target_resolution)
    df = scada_with_criteria.set_index('Date')
    agg = {}
    for col in df.columns:
        if col == '_shift_min':
            agg[col] = 'first'   # el desfase del tramo, no un promedio
        elif col in {'c1','c2','c3','c4','c5','c6','c7','c8','c9','all_valid'}:
            agg[col] = 'min'
        elif col in rules:
            agg[col] = rules[col]
        elif pd.api.types.is_numeric_dtype(df[col]):
            agg[col] = 'mean'
        else:
            agg[col] = 'first'   # columnas de texto (p.ej. *_flag) no se promedian
    out = df.resample(freq, label='left', closed='left').agg(agg).reset_index()
    for k in ('c1','c2','c3','c4','c5','c6','c7','c8','c9','all_valid'):
        if k in out.columns:
            out[k] = out[k].fillna(0).astype(bool)
    return out


def compute_pr_test(scada_with_crit, pvsyst, cfg):
    df = scada_with_crit.merge(pvsyst, on='Date', how='inner', suffixes=('', '_pv'))
    # El SCADA venia desplazado -shift para casar el bucket con PVsyst; aqui se
    # deshace y todo pasa a HORA REAL. Neto: el desfase queda aplicado a la
    # esperada y a la garantizada (que es como se valido), y las fechas que salen
    # -dias, perfil horario, ts de cada intervalo- son las del SCADA.
    if '_shift_min' in df.columns:
        df['Date'] = df['Date'] + pd.to_timedelta(df['_shift_min'].fillna(0), unit='m')
        df = (df.drop(columns=['_shift_min'])
                .sort_values('Date').reset_index(drop=True))
        # En la costura entre dos tramos de T08 (p.ej. el arranque de la regla de
        # verano) el desfase da un salto, y volver a hora real deja de ser
        # inyectivo: la hora del solape aparece dos veces. Es el mismo solape que
        # en un cambio de hora, cae de madrugada, y hay que quitarlo o el MERGE de
        # pr_results falla (DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW).
        _dup = int(df['Date'].duplicated().sum())
        if _dup:
            df = df.drop_duplicates(subset='Date', keep='first').reset_index(drop=True)
            print(f'  [time_shift] costura de tramo T08: {_dup} intervalo(s) '
                  f'solapado(s) descartado(s)')
    pv_col = 'E_Grid' if 'E_Grid' in pvsyst.columns else next(
        (c for c in pvsyst.columns if c != 'Date' and pd.api.types.is_numeric_dtype(pvsyst[c])), None
    )
    guar_series = _guar_factor_series(df['Date'], cfg['contract'])
    dt_h = (df['Date'].iloc[1] - df['Date'].iloc[0]).total_seconds() / 3600 if len(df) >= 2 else 1.0

    _egrid = cfg['contract'].get('energy_ref_sensor') or 'E_GRID'
    if _egrid not in df.columns:
        _egrid = next((c for c in df.columns if c.upper().startswith('E_GRID')), _egrid)
    # La fuente SCADA renombro la energia a la columna 'E_GRID' el 2026-06-26; el
    # sensor mapeado (E_GRID_01/AP_01) queda null desde entonces. Reconstruir la
    # serie: usar el sensor mapeado y, donde falte, la columna E_GRID.
    _meas = df[_egrid].copy() if _egrid in df.columns else pd.Series(0.0, index=df.index)
    if 'E_GRID' in df.columns and _egrid != 'E_GRID':
        _meas = _meas.where(_meas.notna(), df['E_GRID'])
    df['measured_kWh']   = _meas.fillna(0) * dt_h
    df['expected_kWh']   = (_col(df, pv_col) if pv_col else pd.Series(0.0, index=df.index)) * dt_h
    df['guaranteed_kWh'] = df['expected_kWh'] * guar_series
    _v = df['all_valid'].astype(int)
    df['measured_valid_kWh']   = df['measured_kWh']   * _v
    df['guaranteed_valid_kWh'] = df['guaranteed_kWh'] * _v
    df['expected_valid_kWh']   = df['expected_kWh']   * _v

    valid = df[df['all_valid']]
    tm = float(valid['measured_kWh'].sum())
    te = float(valid['expected_kWh'].sum())
    tg = float(valid['guaranteed_kWh'].sum())

    df['_day'] = df['Date'].dt.date

    # Detect optional irradiance columns available after SCADA–PVSyst merge
    _pv_poa = next((c for c in ['POA', 'G_POA', 'G_PoaEff'] if c in df.columns), None)
    _pv_ghi = next((c for c in ['GHI', 'G_Gh', 'G_GlobHor'] if c in df.columns), None)
    _sc_ghi = next((c for c in ['GHI_avg_filt', 'GHI_avg'] if c in df.columns), None)

    _daily_agg = dict(
        measured_kWh         = ('measured_kWh',  'sum'),
        expected_kWh         = ('expected_kWh',  'sum'),
        guaranteed_kWh       = ('guaranteed_kWh','sum'),
        measured_valid_kWh   = ('measured_valid_kWh',   'sum'),
        guaranteed_valid_kWh = ('guaranteed_valid_kWh', 'sum'),
        expected_valid_kWh   = ('expected_valid_kWh',   'sum'),
        valid_intervals      = ('all_valid',     'sum'),
        poa_medida_wm2       = ('POA_avg',       'mean'),
    )
    if _pv_poa:
        _daily_agg['poa_esperada_wm2'] = (_pv_poa, 'mean')
    if _sc_ghi:
        _daily_agg['ghi_medida_wm2'] = (_sc_ghi, 'mean')
    if _pv_ghi:
        _daily_agg['ghi_esperada_wm2'] = (_pv_ghi, 'mean')

    daily = df.groupby('_day').agg(**_daily_agg).reset_index()
    daily['valid_day'] = daily['valid_intervals'] > 0
    daily['delta_pct'] = (
        (daily['measured_valid_kWh'] - daily['guaranteed_valid_kWh'])
        / daily['guaranteed_valid_kWh'].replace(0, float('nan')) * 100
    ).fillna(0)

    _vd = df[df['all_valid']].copy()
    _vd['_hour'] = _vd['Date'].dt.hour
    _hh = _vd.groupby('_hour').agg(
        measured   = ('measured_kWh',   'mean'),
        guaranteed = ('guaranteed_kWh', 'mean'),
    ).reindex(range(24))
    hourly = [{
        'hour': int(h),
        'measured':   (float(r['measured'])   if pd.notna(r['measured'])   else 0.0),
        'guaranteed': (float(r['guaranteed']) if pd.notna(r['guaranteed']) else 0.0),
    } for h, r in _hh.iterrows()]

    # ── Desglose por intervalo: TODAS las magnitudes del Excel/app/reporte ──
    # medida = SCADA procesado (avg_filt); esperada = PVSyst. REF no tiene
    # equivalente PVSyst (queda NaN). Resolución = la del run (15-min u horaria).
    def _pick(_cands):
        return next((_c for _c in _cands if _c in df.columns), None)
    def _icol(_c):
        return df[_c].astype(float).values if (_c and _c in df.columns) else [float('nan')] * len(df)

    sc_poa  = _pick(['POA_avg_filt', 'POA_avg'])
    sc_dhi  = _pick(['DHI_avg_filt', 'DHI_avg'])
    sc_ref  = _pick(['REF_avg_filt', 'REF_avg'])
    sc_tamb = _pick(['T_AMB_avg_filt', 'T_AMB_avg'])
    sc_alb  = _pick(['ALB_avg_filt', 'ALB_avg'])
    pv_dhi  = _pick(['DHI', 'DiffHor', 'DHI_pv'])
    pv_tamb = _pick(['T_AMB_pv', 'T_AMB', 'TAmb', 'T_Amb'])
    pv_alb  = _pick(['ALB_pv', 'ALB', 'Alb'])

    _e_med = df['measured_kWh'].astype(float)
    _e_esp = df['expected_kWh'].astype(float)
    _e_gar = df['guaranteed_kWh'].astype(float)
    _delta_kwh = _e_med - _e_gar
    _delta_pct = (_delta_kwh / _e_gar.replace(0, float('nan')) * 100).fillna(0.0)

    interval_df = pd.DataFrame({
        'ts':                df['Date'].values,
        # Energía (kWh por intervalo)
        'e_medida_kwh':      _e_med.values,
        'e_esperada_kwh':    _e_esp.values,
        'e_garantizada_kwh': _e_gar.values,
        'delta_kwh':         _delta_kwh.values,
        'delta_pct':         _delta_pct.values,
        # Irradiancia (W/m2) medida vs esperada
        'poa_medida_wm2':    _icol(sc_poa),
        'poa_esperada_wm2':  _icol(_pv_poa),
        'ghi_medida_wm2':    _icol(_sc_ghi),
        'ghi_esperada_wm2':  _icol(_pv_ghi),
        'dhi_medida_wm2':    _icol(sc_dhi),
        'dhi_esperada_wm2':  _icol(pv_dhi),
        'ref_medida_wm2':    _icol(sc_ref),
        # Temperatura ambiente (C)
        't_amb_medida_c':    _icol(sc_tamb),
        't_amb_esperada_c':  _icol(pv_tamb),
        # Albedo (adimensional)
        'alb_medida':        _icol(sc_alb),
        'alb_esperada':      _icol(pv_alb),
        # Validez del intervalo (all_valid)
        'valid':             df['all_valid'].astype(bool).values,
    })
    # Criterios c1..c9 por intervalo (bool): mismos que muestra la auditoría
    for _ck in ('c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9'):
        interval_df[_ck] = (df[_ck].astype(bool).values
                            if _ck in df.columns else [False] * len(df))
    return {
        'passed':               bool(tm >= tg),
        'total_measured_kwh':   tm,
        'total_expected_kwh':   te,
        'total_guaranteed_kwh': tg,
        'delta_kwh':            tm - tg,
        'delta_pct':            (tm - tg) / tg * 100 if tg else 0,
        'ratio':                tm / tg if tg else 0,
        'period_start':         str(df['Date'].iloc[0]),
        'period_end':           str(df['Date'].iloc[-1]),
        'daily':                daily.to_dict(orient='records'),
        'hourly':               hourly,
        'interval_df':          interval_df,
    }

## 4 · Apply criteria, aggregate & run PR test

In [12]:
rules      = cfg.get('aggregation_rules', {})
# aggregate_by_hour=True -> forzar 1 h (SCADA y PVsyst se agregan a horario).
# Si no: resolucion mas fina comun -> 15 min si ambos lo permiten, y 1 h solo
# si SCADA o PVsyst ya vienen en horario (no queda mas remedio).
if cfg['contract'].get('aggregate_by_hour'):
    target_res = max(pd.Timedelta(hours=1), scada_res, pvsyst_res)
    print(f'Target resolution: {target_res} (forzado a 1h por aggregate_by_hour)')
else:
    target_res = max(scada_res, pvsyst_res)
    print(f'Target resolution: {target_res}')

scada_with_crit, audit = evaluate_criteria(scada, cfg)
# Desfase T08: se RESTA al SCADA (= sumarselo a PVsyst) una vez evaluados los
# criterios, y solo para que el bucket horario case con PVsyst. compute_pr_test
# devuelve las fechas a hora real despues del merge.
if '_shift_min' in scada_with_crit.columns:
    scada_with_crit['Date'] = (scada_with_crit['Date']
                               - pd.to_timedelta(scada_with_crit['_shift_min'], unit='m'))
    scada_with_crit = scada_with_crit.sort_values('Date').reset_index(drop=True)
_act = sorted(cfg.get('active_criteria', set()), key=lambda _c: int(_c[1:]))
_fz  = {f'c{_i}': getattr(audit, f'c{_i}_failed') for _i in range(1, 10)}
_act_txt = ', '.join(f'{_c}={_fz[_c]:,}' for _c in _act) if _act else '(ninguno)'
_s0, _s1 = scada['Date'].min(), scada['Date'].max()
_p0, _p1 = pvsyst['Date'].min(), pvsyst['Date'].max()
_m0, _m1 = max(_s0, _p0), min(_s1, _p1)
print(
    f'Criteria: valid {audit.valid_intervals:,}/{audit.total_intervals:,} | '
    f'days {audit.valid_days} | valid hours: {audit.aggregated_valid_hours:.1f} h'
)
print(f'  Criterios activos (intervalos NO validos por criterio): {_act_txt}')
print(f'  Periodo casado SCADA - PVsyst: {_m0} -> {_m1}')

if scada_res < target_res:
    scada_with_crit = aggregate_to_target(scada_with_crit, target_res, rules)
    print(f'Aggregated SCADA to {len(scada_with_crit):,} rows @ {target_res}')

if pvsyst_res != target_res:
    freq   = pd.tseries.frequencies.to_offset(target_res)
    agg_pv = {c: (rules[c] if c in rules else ('mean' if pd.api.types.is_numeric_dtype(pvsyst[c]) else 'first')) for c in pvsyst.columns if c != 'Date'}
    pvsyst = pvsyst.set_index('Date').resample(freq, label='left', closed='left').agg(agg_pv).reset_index()

result  = compute_pr_test(scada_with_crit, pvsyst, cfg)
verdict = 'PASS' if result['passed'] else 'FAIL'
print(f"\n{verdict}  Delta = {result['delta_pct']:+.2f}%  ({result['delta_kwh']/1000:+,.0f} MWh)  Ratio = {result['ratio']:.4f}")

## 5 · Render HTML report

In [13]:
import re

# El informe HTML ya NO se genera en pr_runner. Se genera a demanda en el
# notebook `report_export` (y la app PR-EPI_Controller lo descarga con el botón
# "Exportar HTML"). Aquí solo conservamos base_name para registrar la ruta
# esperada del informe en pr_results y en el exit del notebook.
safe_name = re.sub(r'[\\/*?:"<>|]', '_', cfg['plant']['name'])
base_name = f'{safe_name}_PR_Report'


## 6 · Escribir `pr_results` (desglose por intervalo)

In [ ]:
# ── pr_results = desglose por intervalo (cuartohorario/horario) ──
# pr_runner es el UNICO que escribe pr_results. Una fila por (plant_code, ts);
# MERGE: sobreescribe lecturas existentes y anade nuevas. Resolucion = la del run
# (15-min si SCADA y PVSyst lo permiten; si no, 1h). Acumulados/veredicto y los
# conteos de criterios se derivan de esta tabla al leer (agregando).
from pyspark.sql.functions import current_timestamp as _cts, hour as _hourf, to_date as _todate

# autoMerge: si la tabla existe con menos columnas, se le anaden las nuevas.
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

_ivdf = result['interval_df'].copy()
_ivdf['ts'] = pd.to_datetime(_ivdf['ts'])
_ivdf.insert(0, 'plant_name', cfg['plant']['name'])
_ivdf.insert(0, 'plant_code', plant_code)
_ivdf.insert(0, 'run_id', run_id)
# Criterios activos de la planta (mismo valor en todas las filas; lo usa la app/report)
_ivdf['active_criteria'] = ','.join(sorted(cfg.get('active_criteria', set()) or []))

_iv_sp = (spark.createDataFrame(_ivdf)
          .withColumn('day',  _todate('ts'))
          .withColumn('hour', _hourf('ts'))
          .withColumn('executed_at', _cts()))

if not spark.catalog.tableExists('pr_results'):
    _iv_sp.write.format('delta').option('mergeSchema', 'true').saveAsTable('pr_results')
    print(f'pr_results creada (intervalos): {_iv_sp.count():,} filas')
else:
    from delta.tables import DeltaTable
    _t = DeltaTable.forName(spark, 'pr_results')
    (_t.alias('t')
       .merge(_iv_sp.alias('s'), 't.plant_code = s.plant_code AND t.ts = s.ts')
       .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
    print(f'pr_results actualizada (MERGE): {_iv_sp.count():,} intervalos | '
          f'total: {spark.table("pr_results").count():,}')


## 7 · Exit

In [15]:
notebookutils.notebook.exit(json.dumps({
    'html_file':  f'{base_name}.html',
    'verdict':    verdict,
    'delta_pct':  round(result['delta_pct'], 2),
    'ratio':      round(result['ratio'], 4),
    'plant_code': plant_code,
}))